<a href="https://colab.research.google.com/github/huiningjiao02-ship-it/EEG-based-Emotion-Recognition-Aligned-with-IEEE-TAFFC-2025-LibEER-Benchmark-/blob/main/%E8%AE%BA%E6%96%87%E5%A4%8D%E7%8E%B0Comprehensive_Experimental_Benchmark_and_Open_Source_Algorithm_Library_for_EEG_based_Emotion_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# Google Colab: EEG 情绪识别快速复现 (基于 Kaggle 公开数据集)
# 数据集: EEG Brainwave Dataset: Feeling Emotions
# ==============================================================================

# Install necessary libraries
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install kaggle -q

# Import libraries
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from google.colab import files

# Download dataset
print("\n--- Downloading dataset... ---")
!kaggle datasets download -d birdy654/eeg-brainwave-dataset-feeling-emotions
!unzip -q eeg-brainwave-dataset-feeling-emotions.zip -d ./data/
print("Dataset download complete!")

# 5. Data loading and preprocessing
print("\n--- Preprocessing data... ---")
df = pd.read_csv('./data/emotions.csv')

# Separate features (X) and labels (y)
X = df.drop('label', axis=1).values
y = df['label'].values

# Encode labels: NEGATIVE, NEUTRAL, POSITIVE -> 0, 1, 2
le = LabelEncoder()
y = le.fit_transform(y)
print(f"Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to PyTorch tensors (add channel dimension)
X_train = torch.FloatTensor(X_train).unsqueeze(1)
X_test = torch.FloatTensor(X_test).unsqueeze(1)
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

# 6. Define CNN model (simplified DGCNN architecture concept)
class EEG_CNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(EEG_CNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.MaxPool1d(2)
        )

        # Automatically compute input dimension for fully connected layer
        with torch.no_grad():
            dummy = torch.randn(1, 1, input_dim)
            dummy = self.conv_layers(dummy)
            fc_in_dim = dummy.view(1, -1).size(1)

        self.fc_layers = nn.Sequential(
            nn.Linear(fc_in_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

# 7. Training configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = EEG_CNN(input_dim=X_train.shape[2], num_classes=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001)

# Data loaders
batch_size = 32
train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
test_dataset = torch.utils.data.TensorDataset(X_test, y_test)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\n--- Starting training (using device: {device}) ---")

# 8. Training loop
epochs = 40
best_acc = 0.0

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    # Training phase
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    # Validation phase
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)

    # Save best model
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), 'eeg_emotion_best.pth')

    print(f"Epoch [{epoch+1:2d}/{epochs}] -> Loss: {epoch_loss:.4f} | Acc: {acc:.4f} | Best: {best_acc:.4f}")

print("\n--- Training complete! ---")


# 9. Final evaluation
model.load_state_dict(torch.load('eeg_emotion_best.pth'))
model.eval()
all_preds = []
with torch.no_grad():
    for inputs, _ in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())

print("\nClassification report:")
print(classification_report(y_test, all_preds, target_names=le.classes_))


--- Downloading dataset... ---
Dataset URL: https://www.kaggle.com/datasets/birdy654/eeg-brainwave-dataset-feeling-emotions
License(s): copyright-authors
100% 11.9M/11.9M [00:00<00:00, 66.0MB/s]

Dataset download complete!

--- Preprocessing data... ---
Label mapping: {'NEGATIVE': np.int64(0), 'NEUTRAL': np.int64(1), 'POSITIVE': np.int64(2)}
Training set shape: torch.Size([1705, 1, 2548])
Test set shape: torch.Size([427, 1, 2548])

--- Starting training (using device: cuda) ---
Epoch [ 1/40] -> Loss: 1.3798 | Acc: 0.9415 | Best: 0.9415
Epoch [ 2/40] -> Loss: 0.2653 | Acc: 0.9789 | Best: 0.9789
Epoch [ 3/40] -> Loss: 0.0974 | Acc: 0.9859 | Best: 0.9859
Epoch [ 4/40] -> Loss: 0.0884 | Acc: 0.9672 | Best: 0.9859
Epoch [ 5/40] -> Loss: 0.1014 | Acc: 0.9836 | Best: 0.9859
Epoch [ 6/40] -> Loss: 0.0400 | Acc: 0.9906 | Best: 0.9906
Epoch [ 7/40] -> Loss: 0.0701 | Acc: 0.9953 | Best: 0.9953
Epoch [ 8/40] -> Loss: 0.0261 | Acc: 0.9883 | Best: 0.9953
Epoch [ 9/40] -> Loss: 0.0568 | Acc: 0.9906 